#Leer el dataset en formato excel

In [ ]:
import pandas as pd

df_raw = pd.read_excel("Afluencia_Metro_2024.xlsx", header=1, dtype=str)

#Configurar las caracteristicas iniciales de nuestro dataset

In [ ]:
# Renombrar las dos primeras columnas fijas
df_raw = df_raw.rename(columns={
    df_raw.columns[0]: "fecha",
    df_raw.columns[1]: "linea"
})

print(df_raw.columns.tolist())

In [ ]:
#Eliminamos la columna total de pasajeros por día porqué no es relevante a nuestro problema
df_raw = df_raw.drop(columns=["Total general (Número de pasajeros)"], errors='ignore')

In [ ]:
#Limpiar la columna fecha
df_raw["fecha"] = pd.to_datetime(
    df_raw["fecha"],
    format="%d.%m.%Y",
    errors="coerce"
)

# Eliminar filas completamente vacías que Excel a veces inserta
df_raw = df_raw.dropna(subset=["fecha", "linea"])

In [ ]:
# Renombrar columnas de hora: datetime.time(4, 0) → 4
import datetime

df_raw.columns = [
    col.hour if isinstance(col, datetime.time) else col
    for col in df_raw.columns
]

print(df_raw.columns.tolist())

#Número de muestras y variables actuales (4294, 22)

In [ ]:
df_raw.shape

# Identificación de datos por columna

In [ ]:
df_raw.isnull().sum()

#Transformación de nuestro dataset
1. Identificamos que para nuestro problema es importante tener unas columnas especificas como el día de la semana y mes del registro, ya que la afluencia en la red puede variar con esto. El año no lo implementaremos ya que es el mismo para todos y no nos entrega información.

In [ ]:
# Extraer variables temporales desde la columna fecha
df_raw["mes"] = df_raw["fecha"].dt.month # 1 = Enero, 12 = Diciembre
df_raw["dia_semana"] = df_raw["fecha"].dt.dayofweek  # 0=lunes, 6=domingo

# Verificación
print(f"\nMeses únicos: {sorted(df_raw['mes'].unique())}")
print(f"Días únicos:  {sorted(df_raw['dia_semana'].unique())}")

La codificación de nuestras fechas quedo con el siguiente formato:

dia_semana = {0:"Lunes", 1:"Martes", 2:"Miércoles", 3:"Jueves",
                4:"Viernes", 5:"Sábado", 6:"Domingo"}

mes = {1:"Enero", 2:"Febrero", 3:"Marzo", 4:"Abril",
                 5:"Mayo", 6:"Junio", 7:"Julio", 8:"Agosto",
                 9:"Septiembre", 10:"Octubre", 11:"Noviembre", 12:"Diciembre"}

2. Decidimos agregar la columna "festivo", ya que es importante junto con las fechas para la afluencia del metro, además nos servirá mas adelante con el manejo de los valores nulos.

In [ ]:
!pip install holidays

In [ ]:
import holidays

# Festivos colombianos 2024
festivos_co = holidays.Colombia(years=2024)

# Crear columna festivo
df_raw["festivo"] = df_raw["fecha"].dt.date.map(
    lambda x: 1 if x in festivos_co else 0
)

# Verificación
print(f"Días festivos detectados en el dataset: {df_raw['festivo'].sum()}")
print(f"\nFechas marcadas como festivo:")

print(df_raw[df_raw["festivo"] == 1]["fecha"].drop_duplicates().sort_values())

La librería holidays nos da los festivos en Colombia al año 2024, al analizar los resultados vimos hubieron dos festivos del 2024 que no registro por lo que decidimos agregarlos de manera manual. (24 y 31 de Marzo de 2024)

In [ ]:
festivos_manuales = ["2024-03-24", "2024-03-31"]

df_raw["festivo"] = df_raw.apply(
    lambda row: 1 if str(row["fecha"].date()) in festivos_manuales else row["festivo"],
    axis=1
)

# Verificación
print(df_raw[df_raw["festivo"] == 1]["fecha"].drop_duplicates().sort_values())

In [ ]:
#Ahora sí eliminamos la columna fecha que ya no es necesaria
df_raw = df_raw.drop(columns=["fecha"])

3. Como lo que buscamos predecir es la afluencia para una línea del metro y hora especifica, lo que decidimos hacer es formatear estas columnas de hora como una sola y crear los registros correspondientes a cada registro por línea y hora real.

In [ ]:
# Identificar columnas de hora (4 al 23)
hora_cols = [c for c in df_raw.columns if isinstance(c, int)]

# Unpivot: de formato ancho a formato largo
df_long = df_raw.melt(
    id_vars=["linea", "mes", "dia_semana", "festivo"],    # columnas que se mantienen fijas
    value_vars=hora_cols,                                 # columnas que se convierten en filas
    var_name="hora",                                      # nombre de la nueva columna categórica
    value_name="pasajeros"                                # nombre de la columna de valores
)

# Verificación
print(f"Filas antes: {len(df_raw)}")
print(f"Filas después: {len(df_long)}")
print(f"\nNulos en pasajeros: {df_long['pasajeros'].isna().sum()}")

print("\nDataset actual")
df_long.head()

Con esta modificación ahora nuestro dataset cuenta con 85880 registros y 6 columnas.

Y que tenemos en total 6138 registros nulos

4. Al analizar el funcionamiento en horarios del sistema metro, identificamos que cada línea tiene horarios diferentes para cada día, por lo tanto, los valores nulos de cada línea por fuera de su horario de funcionamiento decidimos eliminarlos, ya que no son registros validos debido a que la línea no estaba operando.

Horarios lineas A - B - TA - H - J - K - M - O - P - 1 - 2

Lunes - Sabado: 4:30am - 11:00pm

Horarios lineas A - B - TA - O - 1 - 2

Domingos - Festivos: 5:00 am - 10pm

Horarios lineas K - H - J - M - P

Domingos - Festivos: 9:00 am - 10pm

Horarios linea L - 9:00am - 6:00pm (Fijo a esa fecha)


In [ ]:
# Tabla de horarios operativos por línea y tipo de día
# tipo_dia: 0 = lunes-sabado, 1 = domingo/festivo
horarios = {
    # linea: {tipo_dia: (hora_inicio, hora_fin)}
    "LÍNEA 1":  {0: (4, 23), 1: (5, 22)},
    "LÍNEA 2":  {0: (4, 23), 1: (5, 22)},
    "LÍNEA A":  {0: (4, 23), 1: (5, 22)},
    "LÍNEA B":  {0: (4, 23), 1: (5, 22)},
    "LÍNEA T-A":{0: (4, 23), 1: (5, 22)},
    "LÍNEA O":  {0: (4, 23), 1: (5, 22)},
    "LÍNEA H":  {0: (4, 23), 1: (9, 22)},
    "LÍNEA J":  {0: (4, 23), 1: (9, 22)},
    "LÍNEA K":  {0: (4, 23), 1: (9, 22)},
    "LÍNEA M":  {0: (4, 23), 1: (9, 22)},
    "LÍNEA P":  {0: (4, 23), 1: (9, 22)},
    "LÍNEA L":  {0: (9, 18), 1: (9, 17)},  # horario fijo todos los días
}

def es_fuera_de_horario(row):
    tipo_dia = 1 if (row["dia_semana"] == 6 or row["festivo"] == 1) else 0
    h_inicio, h_fin = horarios[row["linea"]][tipo_dia]
    return row["hora"] < h_inicio or row["hora"] > h_fin

# Mask: nulo Y fuera de horario
mask_eliminar = (
    df_long["pasajeros"].isna() &
    df_long.apply(es_fuera_de_horario, axis=1)
)

print(f"Filas antes:          {len(df_long)}")
print(f"Filas a eliminar:     {mask_eliminar.sum()}")

df_long = df_long[~mask_eliminar].reset_index(drop=True)

print(f"Filas después:        {len(df_long)}")
print(f"Nulos restantes:      {df_long['pasajeros'].isna().sum()}")

Luego de esta primera eliminación de valores nulos tenemos un total 80655 registros y quedan en total 913 valores nulos

5. Para los valores nulos que nos quedan que estan en horarios habituales de trabajo del metro, vamos a hacer una exploración de la distribución de estos y decidir si los imputamos con la media o la mediana

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("DISTRIBUCIÓN DE NULOS\n")

print("Por línea:")
print(df_long[df_long["pasajeros"].isna()].groupby("linea").size().sort_values(ascending=False))

print("\nPor hora:")
print(df_long[df_long["pasajeros"].isna()].groupby("hora").size().sort_values(ascending=False))

print("\nPor día de semana:")
print(df_long[df_long["pasajeros"].isna()].groupby("dia_semana").size().sort_values(ascending=False))

df_long["pasajeros"] = pd.to_numeric(df_long["pasajeros"], errors="coerce")

print("\nESTADÍSTICAS GENERALES DE PASAJEROS")
print(df_long["pasajeros"].describe())
print(f"Skewness: {df_long['pasajeros'].skew():.3f}")

# Distribución por línea (boxplot)
plt.figure(figsize=(14, 5))
sns.boxplot(data=df_long, x="linea", y="pasajeros", order=sorted(df_long["linea"].unique()))
plt.title("Distribución de pasajeros por línea")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Dado que la distribución de pasajeros presenta un sesgo positivo alto, con valores extremos que alcanzan los 88.553 pasajeros y una brecha significativa entre la media (4.193) y la mediana (987), se optó por imputar los valores faltantes con la mediana. A diferencia de la media, la mediana es robusta frente a outliers y representa mejor el valor típico en distribuciones asimétricas. Adicionalmente, la imputación se realizó de forma segmentada por línea, hora y día de la semana, ya que la afluencia varía estructuralmente según estos factores. Este enfoque garantiza que cada valor imputado sea coherente con el contexto operativo real del registro.

In [ ]:
df_long["pasajeros"] = df_long.groupby(
    ["linea", "hora", "dia_semana"]
)["pasajeros"].transform(
    lambda x: x.fillna(x.median())
)

print(f"Nulos restantes: {df_long['pasajeros'].isna().sum()}")

Con esta primera imputación por linea, hora y día de la semana, quedan 48 nulos que no tiene valores correspondientes a esta agrupación, por lo tanto haremos una segunda mas reducida

In [ ]:
df_long["pasajeros"] = df_long.groupby(
    ["linea", "hora"]
)["pasajeros"].transform(
    lambda x: x.fillna(x.median())
)

print(f"Nulos restantes: {df_long['pasajeros'].isna().sum()}")

In [ ]:
df_long.info()

6. Ahora vamos a codificar las variables, como notamos las variables dia_semana, mes, festivo ya estan numericos y en rangos pequeños, por lo tanto vamos a decidir dejarlos así y no normalizar su escala. Para las variables hora y linea que son categoricas se aplicaron dos estrategias distintas según la naturaleza de cada variable. La variable hora se trató con Label Encoding (conversión directa a entero), ya que representa una variable ordinal con un orden cronológico natural — las horas tienen una progresión lógica que los modelos pueden aprovechar directamente. Por su parte, la variable linea se codificó con One-Hot Encoding, ya que es una variable nominal donde no existe ningún orden inherente entre las líneas del metro, por lo que asignarle valores enteros introduciría una jerarquía artificial que podría sesgar el modelo.

In [ ]:
#Hora
df_long["hora"] = df_long["hora"].astype(int)

#Linea
df_long = pd.get_dummies(df_long, columns=["linea"], prefix="linea", dtype=int)

# Verificación
print(df_long.shape)
print(df_long.dtypes)
print(df_long.head())

In [21]:
# Guardar como CSV
df_long.to_csv('afluencia_metro_2024_transformado.csv', index=False)
